# **Cleaning Raw Data Based on Audit Findings**

## Cleaning Plan (from audit)

1. **Drop useless columns**
   - `weight` (97% missing)
   - `encounter_id` (unique ID, no predictive value)
   - `patient_nbr` (patient ID, would cause data leakage)

2. **Handle missing values**
   - `race`, `payer_code`, `medical_specialty`, `diag_1`, `diag_2`, `diag_3`: replace `"?"` with `"Unknown"`
   - `max_glu_serum`, `A1Cresult`: fill `NaN` with `"Unknown"`

3. **Convert target to binary**
   - `1` = readmitted within 30 days (`<30`)
   - `0` = not readmitted (`NO` or `>30`)

4. **Convert age from bracket strings to numeric midpoints**

5. **Map coded ID columns to readable labels using `IDS_mapping.csv`**
   - `admission_type_id` → `admission_type`
   - `discharge_disposition_id` → `discharge_disposition`
   - `admission_source_id` → `admission_source`

6. **Drop original columns** replaced by the cleaned versions above


In [17]:
import sys
sys.path.append('..')

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from src.config import RAW_DATA, CLEAN_DATA
from src.data_loader import load_raw_data

# Load data
data = load_raw_data()

Shape: (101766, 50)


## Step 1: Drop Useless Columns

Three columns from the audit are removed immediately:
- **`weight`** - 96.9% missing; imputing ~97k values would introduce noise rather than signal.
- **`encounter_id`** - a unique row identifier with no predictive information.
- **`patient_nbr`** - a patient identifier; since some patients appear multiple times, including it would cause data leakage (the model could memorise patients instead of learning clinical patterns).

In [18]:
# Drop useless columns
cols_to_drop = ['weight', 'encounter_id', 'patient_nbr']
data_clean = data.drop(columns=cols_to_drop)

print(f"Shape after dropping: {data_clean.shape}")
print(f"Dropped: {cols_to_drop}")

Shape after dropping: (101766, 47)
Dropped: ['weight', 'encounter_id', 'patient_nbr']


## Step 2: Handle Missing Values

The audit identified two types of missingness:

**Type A - `"?"` placeholder strings** in categorical columns. These are replaced with `"Unknown"` so the information that the value was not recorded is preserved as its own category (rather than dropped or imputed).

**Type B - genuine `NaN`** in `max_glu_serum` and `A1Cresult`, which are lab tests that were simply not performed for most patients. These are also filled with `"Unknown"` for the same reason.

In [19]:
# Replace "?" placeholder strings
missing_cols = ['race', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3']

for col in missing_cols:
    missing_count = (data_clean[col] == '?').sum()
    data_clean[col] = data_clean[col].replace('?', 'Unknown')
    print(f"{col}: replaced {missing_count} '?' with 'Unknown'")

print("\nDone.")

race: replaced 2273 '?' with 'Unknown'
payer_code: replaced 40256 '?' with 'Unknown'
medical_specialty: replaced 49949 '?' with 'Unknown'
diag_1: replaced 21 '?' with 'Unknown'
diag_2: replaced 358 '?' with 'Unknown'
diag_3: replaced 1423 '?' with 'Unknown'

Done.


In [20]:
# Fill NaN in lab-test columns (test simply not performed)
data_clean['max_glu_serum'] = data_clean['max_glu_serum'].fillna('Unknown')
data_clean['A1Cresult']     = data_clean['A1Cresult'].fillna('Unknown')

# Verify no remaining nulls across the whole dataframe
remaining = data_clean.isnull().sum()
remaining[remaining > 0]

Series([], dtype: int64)

## Step 3: Convert Target Column to Binary

The original `readmitted` column has three categories: `<30`, `>30`, and `NO`. The modelling goal is to predict **30-day readmission**, so:
- `1` = readmitted within 30 days (`<30`)
- `0` = not readmitted within 30 days (`>30` or `NO`)

The original column is kept temporarily and dropped later with the other originals.

In [21]:
print("Before conversion:")
print(data_clean['readmitted'].value_counts())

data_clean['readmitted_binary'] = (data_clean['readmitted'] == '<30').astype(int)

print("\nAfter conversion:")
print(data_clean['readmitted_binary'].value_counts())
print(f"\nReadmitted within 30 days: {data_clean['readmitted_binary'].mean()*100:.1f}%")

Before conversion:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

After conversion:
readmitted_binary
0    90409
1    11357
Name: count, dtype: int64

Readmitted within 30 days: 11.2%


## Step 4: Convert Age to Numeric

Age is stored as decade brackets (`[0-10)`, `[10-20)`, …). These are converted to their midpoint values (5, 15, …, 95) so the column can be used as a continuous feature in models that benefit from numeric input.

In [22]:
print("Current age distribution:")
print(data_clean['age'].value_counts().sort_index())

age_split = data_clean['age'].str.strip('[]()').str.split('-', expand=True)
data_clean['age_numeric'] = age_split.astype(float).mean(axis=1)

print("\nConverted age_numeric distribution:")
print(data_clean['age_numeric'].value_counts().sort_index())

Current age distribution:
age
[0-10)        161
[10-20)       691
[20-30)      1657
[30-40)      3775
[40-50)      9685
[50-60)     17256
[60-70)     22483
[70-80)     26068
[80-90)     17197
[90-100)     2793
Name: count, dtype: int64

Converted age_numeric distribution:
age_numeric
5.0       161
15.0      691
25.0     1657
35.0     3775
45.0     9685
55.0    17256
65.0    22483
75.0    26068
85.0    17197
95.0     2793
Name: count, dtype: int64


> **Observation:** The patient population is heavily concentrated in the 60–80 age range, consistent with a diabetic inpatient cohort.

## Step 5: Map ID Columns to Readable Labels

Three columns store admission/discharge information as numeric codes:
- `admission_type_id` - e.g., 1 = Emergency, 2 = Urgent, 3 = Elective
- `discharge_disposition_id` - e.g., 1 = Discharged to home, 11 = Expired
- `admission_source_id` - e.g., 7 = Emergency Room, 1 = Physician Referral

Leaving them as integers would mislead models into treating them as ordered quantities (3 > 2 > 1) when they are purely categorical. Converting to text labels and later one-hot encoding them is the correct approach.

The `IDS_mapping.csv` file contains all three lookup tables stacked vertically, so we extract each section separately.

In [23]:
mapping_df = pd.read_csv("../Dataset/raw/IDS_mapping.csv")
print("Mapping file loaded. First few rows:")
print(mapping_df.head())

Mapping file loaded. First few rows:
  admission_type_id    description
0                 1      Emergency
1                 2         Urgent
2                 3       Elective
3                 4        Newborn
4                 5  Not Available


In [24]:
# Locate where each section starts
discharge_start = mapping_df[mapping_df['admission_type_id'] == 'discharge_disposition_id'].index[0]
source_start    = mapping_df[mapping_df['admission_type_id'] == 'admission_source_id'].index[0]

print(f"discharge section starts at row: {discharge_start}")
print(f"source section starts at row:    {source_start}")

# Extract admission_type mapping (rows before discharge header)
admit_rows = mapping_df.iloc[:discharge_start].dropna()
admit_map  = dict(zip(admit_rows['admission_type_id'].astype(float).astype(int),
                      admit_rows['description']))

# Extract discharge_disposition mapping
discharge_rows = mapping_df.iloc[discharge_start+1:source_start].dropna()
disp_map = dict(zip(discharge_rows['admission_type_id'].astype(float).astype(int),
                    discharge_rows['description']))

# Extract admission_source mapping
source_rows = mapping_df.iloc[source_start+1:].dropna()
source_map  = dict(zip(source_rows['admission_type_id'].astype(float).astype(int),
                       source_rows['description']))

print(f"\nadmission_type_id:        {len(admit_map)} codes extracted")
print(f"discharge_disposition_id: {len(disp_map)} codes extracted")
print(f"admission_source_id:      {len(source_map)} codes extracted")

discharge section starts at row: 9
source section starts at row:    41

admission_type_id:        7 codes extracted
discharge_disposition_id: 29 codes extracted
admission_source_id:      24 codes extracted


In [25]:
# Apply mappings
data_clean['admission_type']       = data_clean['admission_type_id'].map(admit_map)
data_clean['discharge_disposition'] = data_clean['discharge_disposition_id'].map(disp_map)
data_clean['admission_source']      = data_clean['admission_source_id'].map(source_map)

# Check for unmapped values (will appear as NaN)
print("NaN counts after initial mapping:")
print(f"  admission_type:       {data_clean['admission_type'].isna().sum()}")
print(f"  discharge_disposition:{data_clean['discharge_disposition'].isna().sum()}")
print(f"  admission_source:     {data_clean['admission_source'].isna().sum()}")

NaN counts after initial mapping:
  admission_type:       5291
  discharge_disposition:3691
  admission_source:     6781


### Diagnosing the Unmapped Codes

Some IDs in the data have no corresponding entry in the mapping file. These are codes that the original dataset documentation marks as **NULL** (i.e., the information was not available for that encounter). We need to identify them and add them manually to the dictionaries before re-applying the maps.

In [26]:
# Find which codes in the data are absent from each mapping dictionary
missing_admit  = set(data_clean['admission_type_id'].unique())       - set(admit_map.keys())
missing_disp   = set(data_clean['discharge_disposition_id'].unique()) - set(disp_map.keys())
missing_source = set(data_clean['admission_source_id'].unique())      - set(source_map.keys())

print(f"Unmapped admission_type_id codes:        {sorted(missing_admit)}")
print(f"Unmapped discharge_disposition_id codes: {sorted(missing_disp)}")
print(f"Unmapped admission_source_id codes:      {sorted(missing_source)}")

Unmapped admission_type_id codes:        [np.int64(6)]
Unmapped discharge_disposition_id codes: [np.int64(18)]
Unmapped admission_source_id codes:      [np.int64(17)]


### Fix: Add Missing Codes and Re-apply Mappings

Codes 6 (admission type), 18 (discharge disposition), and 17 (admission source) all correspond to **NULL** per the dataset documentation - the information was simply not recorded. We add them to the dictionaries and re-apply all three mappings in one pass.

In [27]:
# Add the three NULL codes that were absent from the mapping file
admit_map[6]  = 'NULL'
disp_map[18]  = 'NULL'
source_map[17] = 'NULL'

# Re-apply all three mappings with the complete dictionaries
data_clean['admission_type']        = data_clean['admission_type_id'].map(admit_map)
data_clean['discharge_disposition']  = data_clean['discharge_disposition_id'].map(disp_map)
data_clean['admission_source']       = data_clean['admission_source_id'].map(source_map)

# Safety net: any remaining NaN means an unexpected code  -  label it 'Unknown'
data_clean['admission_type']        = data_clean['admission_type'].fillna('Unknown')
data_clean['discharge_disposition']  = data_clean['discharge_disposition'].fillna('Unknown')
data_clean['admission_source']       = data_clean['admission_source'].fillna('Unknown')

# Verify  -  all should be 0
print("NaN counts after fix:")
print(data_clean[['admission_type', 'discharge_disposition', 'admission_source']].isna().sum())
print("\n✓ All ID columns fully mapped.")

NaN counts after fix:
admission_type           0
discharge_disposition    0
admission_source         0
dtype: int64

✓ All ID columns fully mapped.


## Step 6: Drop Original Columns

Now that we have clean replacements for each original column, the originals are removed:

| Original | Replaced by |
|---|---|
| `age` | `age_numeric` |
| `readmitted` | `readmitted_binary` |
| `admission_type_id` | `admission_type` |
| `discharge_disposition_id` | `discharge_disposition` |
| `admission_source_id` | `admission_source` |

In [28]:
cols_to_drop_final = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id',
    'age',
    'readmitted'
]

data_final = data_clean.drop(columns=cols_to_drop_final, errors='ignore')

print(f"Shape after dropping originals: {data_final.shape}")
print(f"Final columns ({len(data_final.columns)}):")
print(data_final.columns.tolist())

Shape after dropping originals: (101766, 47)
Final columns (47):
['race', 'gender', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted_binary', 'age_numeric', 'admission_type', 'discharge_disposition', 'admission_source']


## Save Cleaned Dataset

In [29]:
import os

CLEAN_DATA = "../Dataset/cleaned/cleaned_data.csv"
os.makedirs(os.path.dirname(CLEAN_DATA), exist_ok=True)

data_final.to_csv(CLEAN_DATA, index=False)

print(f"Saved to:  {CLEAN_DATA}")
print(f"Shape:     {data_final.shape}")
print(f"Columns:   {data_final.shape[1]}")

Saved to:  ../Dataset/cleaned/cleaned_data.csv
Shape:     (101766, 47)
Columns:   47


In [30]:
# Quick sanity check  -  reload from disk and confirm shape and no unexpected nulls
df = pd.read_csv("../Dataset/cleaned/cleaned_data.csv")
print(f"Reloaded shape: {df.shape}")
print(f"\nNull counts (should all be 0):")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nFirst 5 rows:")
df.head()

Reloaded shape: (101766, 47)

Null counts (should all be 0):
admission_type           5291
discharge_disposition    3691
admission_source         6781
dtype: int64

First 5 rows:


,race,gender,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,...,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_binary,age_numeric,admission_type,discharge_disposition,admission_source
0,Caucasian,Female,1,Unknown,Pediatrics-Endocrinology,41,0,1,0,0,...,No,No,No,No,No,0,5.0,NaN,Not Mapped,Physician Referral
1,Caucasian,Female,3,Unknown,Unknown,59,0,18,0,0,...,No,No,No,Ch,Yes,0,15.0,Emergency,Discharged to home,Emergency Room
2,AfricanAmerican,Female,2,Unknown,Unknown,11,5,13,2,0,...,No,No,No,No,Yes,0,25.0,Emergency,Discharged to home,Emergency Room
3,Caucasian,Male,2,Unknown,Unknown,44,1,16,0,0,...,No,No,No,Ch,Yes,0,35.0,Emergency,Discharged to home,Emergency Room
4,Caucasian,Male,1,Unknown,Unknown,51,0,8,0,0,...,No,No,No,Ch,Yes,0,45.0,Emergency,Discharged to home,Emergency Room


In [31]:
df.columns

Index(['race', 'gender', 'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted_binary',
       'age_numeric', 'admission_type', 'discharge_disposition',
       'admission_source'],
      dtype='str')